# PnP-ADMM Super-Resolution Inference Notebook
## Part 2: Image Deblurring and Super-Resolution

This notebook:
- Loads a pre-trained DnCNN denoiser model
- Performs super-resolution on uploaded images (screenshots)
- Uses PnP-ADMM algorithm for image restoration

**Setup Instructions:**
1. Upload your trained model file: `dncnn_denoiser_div2k.pth`
2. Upload any image/screenshot you want to enhance
3. Run all cells!

**How to add files to this notebook:**
- Click **"+ Add Data"** → **"Upload"** → Select your files
- Model file and images will appear in `/kaggle/input/`

In [ ]:
# ============================================================
# 1. Imports and Setup
# ============================================================

import os
import glob
import time
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from scipy.ndimage import zoom
from scipy.signal import convolve2d
from scipy.sparse.linalg import cg, LinearOperator

from skimage import io, color, img_as_float
from skimage.metrics import peak_signal_noise_ratio as psnr

print("✓ All imports successful")

In [ ]:
# ============================================================
# 2. Reproducibility and Device
# ============================================================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ============================================================
# 3. Configuration
# ============================================================

# Model architecture (must match training)
MODEL_FEATURES = 64
MODEL_DEPTH = 6

# Super-resolution settings
SR_SCALE = 2              # Upscaling factor (2x)
LR_NOISE_SIGMA = 0.005    # Noise level (REDUCED to reduce artifacts)
ADMM_RHO = 0.005          # ADMM penalty (REDUCED for gentler processing)
ADMM_ITERS = 40           # Number of iterations (INCREASED for better convergence)
CG_ITERS = 30             # CG iterations (INCREASED)

# Denoiser strength control
DENOISER_WEIGHT = 0.5     # Reduce denoiser influence (0-1)

# Maximum image size (to avoid memory issues)
MAX_IMAGE_SIZE = 512      # Will resize if larger

print("Configuration loaded")
print(f"  SR scale: {SR_SCALE}x")
print(f"  ADMM iterations: {ADMM_ITERS}")
print(f"  Max image size: {MAX_IMAGE_SIZE}")

In [ ]:
# ============================================================
# 4. Utility Functions
# ============================================================

def find_all_images():
    """Find all uploaded images in /kaggle/input"""
    valid_exts = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"}
    all_images = []
    
    for root, dirs, files in os.walk("/kaggle/input"):
        for fname in files:
            ext = os.path.splitext(fname)[1].lower()
            if ext in valid_exts:
                all_images.append(os.path.join(root, fname))
    
    return sorted(all_images)


def find_model_file():
    """Find the uploaded .pth model file"""
    pth_files = glob.glob("/kaggle/input/models/nikhilsaravanan/dncnn-denoiser-div2k/pytorch/default/1/tiny_dncnn_div2k_gray.pth", recursive=True)
    if len(pth_files) == 0:
        return None
    return pth_files[0]  # Return first .pth file found


def read_image_gray(path):
    """Read image and convert to grayscale float32 [0, 1]"""
    img = io.imread(path)

    if img.ndim == 2:
        gray = img_as_float(img).astype(np.float32)
    elif img.ndim == 3:
        if img.shape[2] == 4:
            img = color.rgba2rgb(img)
        gray = color.rgb2gray(img)
        gray = img_as_float(gray).astype(np.float32)
    else:
        raise ValueError(f"Unsupported image shape: {img.shape}")

    return np.clip(gray, 0.0, 1.0)


def show_image(title, img, cmap='gray', figsize=(6, 6)):
    """Display a single image"""
    plt.figure(figsize=figsize)
    plt.imshow(img, cmap=cmap, vmin=0, vmax=1)
    plt.title(title, fontsize=14, fontweight='bold')
    plt.axis("off")
    plt.tight_layout()
    plt.show()


print("✓ Utility functions loaded")

In [ ]:
# ============================================================
# 5. Find Uploaded Files
# ============================================================

# Find model file
model_path = find_model_file()
if model_path is None:
    print("⚠ No .pth model file found!")
    print("\nPlease upload your trained model:")
    print("  1. Click '+ Add Data' at top right")
    print("  2. Select 'Upload'")
    print("  3. Upload your dncnn_denoiser_div2k.pth file")
    print("  4. Re-run this cell")
    raise FileNotFoundError("Model file not found. Please upload .pth file.")
else:
    print(f"✓ Found model file: {model_path}")
    print(f"  Size: {os.path.getsize(model_path) / 1024 / 1024:.2f} MB")

# Find images
all_images = find_all_images()
print(f"\n✓ Found {len(all_images)} uploaded image(s)")

if len(all_images) == 0:
    print("\n⚠ No images found!")
    print("\nPlease upload an image/screenshot:")
    print("  1. Click '+ Add Data' at top right")
    print("  2. Select 'Upload'")
    print("  3. Upload your image file")
    print("  4. Re-run this cell")
else:
    print("\nAvailable images:")
    for i, img_path in enumerate(all_images):
        print(f"  [{i}] {os.path.basename(img_path)}")
        print(f"      {img_path}")

In [ ]:
# ============================================================
# 6. DnCNN Model Definition
# ============================================================

class TinyDnCNN(nn.Module):
    """Lightweight DnCNN for grayscale denoising"""
    def __init__(self, channels=1, features=64, depth=6):
        super().__init__()
        layers = []
        
        # First layer
        layers.append(nn.Conv2d(channels, features, kernel_size=3, padding=1))
        layers.append(nn.ReLU(inplace=True))

        # Middle layers
        for _ in range(depth):
            layers.append(nn.Conv2d(features, features, kernel_size=3, padding=1))
            layers.append(nn.BatchNorm2d(features))
            layers.append(nn.ReLU(inplace=True))

        # Final layer
        layers.append(nn.Conv2d(features, channels, kernel_size=3, padding=1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        # Residual learning: predict noise and subtract
        return x - self.net(x)


print("✓ Model architecture defined")

In [ ]:
# ============================================================
# 7. Load Pretrained Model
# ============================================================

model = TinyDnCNN(channels=1, features=MODEL_FEATURES, depth=MODEL_DEPTH).to(device)

# Load weights
try:
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    print(f"✓ Successfully loaded model from: {model_path}")
    print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
except Exception as e:
    print(f"❌ Error loading model: {e}")
    print("\nMake sure:")
    print("  1. The model file matches the architecture")
    print("  2. The file is not corrupted")
    raise

In [ ]:
# ============================================================
# 8. Forward Model for Super-Resolution
# ============================================================

def gaussian_kernel(size=7, sigma=1.6):
    """Create Gaussian blur kernel"""
    ax = np.arange(-(size // 2), size // 2 + 1)
    xx, yy = np.meshgrid(ax, ax)
    kernel = np.exp(-(xx**2 + yy**2) / (2.0 * sigma**2))
    kernel /= kernel.sum()
    return kernel.astype(np.float32)


BLUR_KERNEL = gaussian_kernel(size=7, sigma=1.6)


def blur_image(x, kernel=BLUR_KERNEL):
    """Apply Gaussian blur"""
    return convolve2d(x, kernel, mode="same", boundary="symm")


def downsample(x, scale):
    """Downsample by taking every scale-th pixel"""
    return x[::scale, ::scale]


def upsample_zero(y, scale, target_shape):
    """Upsample by zero-insertion"""
    out = np.zeros(target_shape, dtype=np.float32)
    out[::scale, ::scale] = y
    return out


def forward_SR(x, scale, kernel=BLUR_KERNEL):
    """Forward model: blur + downsample"""
    xb = blur_image(x, kernel)
    y = downsample(xb, scale)
    return y.astype(np.float32)


def adjoint_SR(y, target_shape, scale, kernel=BLUR_KERNEL):
    """Adjoint operator: upsample + transpose blur"""
    up = upsample_zero(y, scale, target_shape)
    kernel_flip = np.flipud(np.fliplr(kernel))
    out = convolve2d(up, kernel_flip, mode="same", boundary="symm")
    return out.astype(np.float32)


print("✓ Forward model functions loaded")

In [ ]:
# ============================================================
# 9. x-update using Conjugate Gradient
# ============================================================

def x_update_cg(y_lr, target_shape, scale, rho, z, sigma, kernel=BLUR_KERNEL, n_cg=25):
    """Solve data fidelity subproblem using conjugate gradient"""
    h, w = target_shape
    n = h * w

    # Compute right-hand side
    At_y = adjoint_SR(y_lr, target_shape, scale, kernel)
    b = (1.0 / (sigma**2)) * At_y + rho * z

    # Define matrix-vector product
    def matvec(x_vec):
        x_img = x_vec.reshape(target_shape)
        Ax = forward_SR(x_img, scale, kernel)
        AtAx = adjoint_SR(Ax, target_shape, scale, kernel)
        out = (1.0 / (sigma**2)) * AtAx + rho * x_img
        return out.ravel()

    Aop = LinearOperator((n, n), matvec=matvec, dtype=np.float32)
    x0 = z.ravel().astype(np.float32)
    b_vec = b.ravel().astype(np.float32)

    # Solve using conjugate gradient
    x_sol, info = cg(Aop, b_vec, x0=x0, maxiter=n_cg, atol=1e-6)
    if info != 0 and info > 0:
        print(f"  CG: Did not fully converge (iter={info})")

    return x_sol.reshape(target_shape).astype(np.float32)


print("✓ CG solver loaded")

In [ ]:
# ============================================================
# 10. PnP-ADMM Super-Resolution Algorithm
# ============================================================

def pnp_admm_sr(
    y_lr,
    target_shape,
    scale,
    sigma,
    model,
    rho=0.01,
    n_iter=30,
    n_cg=25,
    gt=None,
    verbose=True
):
    """PnP-ADMM for super-resolution"""
    model.eval()

    # Initialize with bicubic upsampling
    x = zoom(y_lr, scale, order=3)
    x = x[:target_shape[0], :target_shape[1]].astype(np.float32)

    if x.shape != target_shape:
        out = np.zeros(target_shape, dtype=np.float32)
        hh = min(x.shape[0], target_shape[0])
        ww = min(x.shape[1], target_shape[1])
        out[:hh, :ww] = x[:hh, :ww]
        x = out

    v = x.copy()
    u = np.zeros_like(x, dtype=np.float32)

    history = {
        "psnr": [],
        "data_fidelity": [],
        "residual": [],
        "iter_time": []
    }

    if verbose:
        print("\n" + "=" * 60)
        print(f"  PnP-ADMM Super-Resolution (×{scale})")
        print("=" * 60)
        print(f"  ρ = {rho}, iterations = {n_iter}, CG steps = {n_cg}")
        print(f"  Input LR shape: {y_lr.shape}")
        print(f"  Target HR shape: {target_shape}")
        print("=" * 60 + "\n")

    for k in range(n_iter):
        iter_start = time.time()

        # x-update (data fidelity via CG)
        z = v - u
        x = x_update_cg(
            y_lr=y_lr,
            target_shape=target_shape,
            scale=scale,
            rho=rho,
            z=z,
            sigma=sigma,
            kernel=BLUR_KERNEL,
            n_cg=n_cg
        )

        # v-update (denoiser prior)
        v_input = np.clip(x + u, 0, 1)
        with torch.no_grad():
            t_in = torch.from_numpy(v_input).float().unsqueeze(0).unsqueeze(0).to(device)
            t_out = model(t_in)
        v = np.clip(t_out.squeeze().cpu().numpy(), 0, 1).astype(np.float32)

        # u-update (dual variable)
        u = u + x - v

        # Compute metrics
        iter_time = time.time() - iter_start
        data_term = 0.5 / (sigma**2) * np.sum((forward_SR(x, scale, BLUR_KERNEL) - y_lr) ** 2)
        residual = np.sqrt(np.sum((x - v) ** 2))

        history["data_fidelity"].append(float(data_term))
        history["residual"].append(float(residual))
        history["iter_time"].append(float(iter_time))

        if gt is not None:
            current_psnr = psnr(gt, np.clip(x, 0, 1), data_range=1.0)
            history["psnr"].append(float(current_psnr))

            if verbose and (k < 3 or k % 5 == 0 or k == n_iter - 1):
                print(
                    f"  Iter {k+1:2d}/{n_iter} | PSNR: {current_psnr:.2f} dB | "
                    f"||x-v||: {residual:.4f} | Time: {iter_time:.2f}s"
                )
        else:
            if verbose and (k < 3 or k % 5 == 0 or k == n_iter - 1):
                print(
                    f"  Iter {k+1:2d}/{n_iter} | "
                    f"||x-v||: {residual:.4f} | Time: {iter_time:.2f}s"
                )

    if verbose:
        print("=" * 60 + "\n")

    return np.clip(x, 0, 1), history


print("✓ PnP-ADMM algorithm loaded")

In [ ]:
# ============================================================
# 11. Select and Load Test Image
# ============================================================

if len(all_images) == 0:
    print("❌ No images available. Please upload an image first.")
else:
    # Use the first image by default
    # Change this index if you want to use a different image
    test_image_idx = 0
    
    test_image_path = all_images[test_image_idx]
    print(f"Selected image: {os.path.basename(test_image_path)}")
    print(f"Full path: {test_image_path}")
    
    # Load image
    img = read_image_gray(test_image_path)
    print(f"\nOriginal image shape: {img.shape}")
    
    # Resize if too large
    if img.shape[0] > MAX_IMAGE_SIZE or img.shape[1] > MAX_IMAGE_SIZE:
        scale_h = MAX_IMAGE_SIZE / img.shape[0]
        scale_w = MAX_IMAGE_SIZE / img.shape[1]
        resize_factor = min(scale_h, scale_w)
        img = zoom(img, resize_factor, order=3).astype(np.float32)
        img = np.clip(img, 0, 1)
        print(f"Resized to: {img.shape} (to avoid memory issues)")
    
    # Show the image
    show_image("Original Image (HR Ground Truth)", img)

In [ ]:
# ============================================================
# 12. Create Degraded LR Image
# ============================================================

# Adjust dimensions to be divisible by scale
H, W = img.shape
H_adj = (H // SR_SCALE) * SR_SCALE
W_adj = (W // SR_SCALE) * SR_SCALE
img = img[:H_adj, :W_adj]

print(f"Adjusted image shape: {img.shape}")

# Create low-resolution image (blur + downsample + noise)
np.random.seed(0)
y_lr_clean = forward_SR(img, SR_SCALE, BLUR_KERNEL)
noise = np.random.randn(*y_lr_clean.shape).astype(np.float32) * LR_NOISE_SIGMA
y_lr = np.clip(y_lr_clean + noise, 0, 1)

print(f"\nDegradation info:")
print(f"  HR shape: {img.shape}")
print(f"  LR shape: {y_lr.shape}")
print(f"  Noise sigma: {LR_NOISE_SIGMA}")

# Bicubic baseline
bicubic = zoom(y_lr, SR_SCALE, order=3)
bicubic = bicubic[:img.shape[0], :img.shape[1]]
bicubic = np.clip(bicubic, 0, 1)

psnr_bicubic = psnr(img, bicubic, data_range=1.0)
print(f"\nBicubic baseline PSNR: {psnr_bicubic:.2f} dB")

# Visualize degradation
fig, ax = plt.subplots(1, 3, figsize=(15, 5))

ax[0].imshow(img, cmap="gray", vmin=0, vmax=1)
ax[0].set_title(f"Ground Truth HR\n{img.shape[1]}×{img.shape[0]}", fontsize=12, fontweight='bold')
ax[0].axis("off")

ax[1].imshow(y_lr, cmap="gray", vmin=0, vmax=1)
ax[1].set_title(f"Observed LR (degraded)\n{y_lr.shape[1]}×{y_lr.shape[0]}", fontsize=12, fontweight='bold')
ax[1].axis("off")

ax[2].imshow(bicubic, cmap="gray", vmin=0, vmax=1)
ax[2].set_title(f"Bicubic Upsampling\nPSNR: {psnr_bicubic:.2f} dB", fontsize=12, fontweight='bold')
ax[2].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 13. Run PnP-ADMM Super-Resolution
# ============================================================

print("\nStarting PnP-ADMM super-resolution...\n")

x_hat, history = pnp_admm_sr(
    y_lr=y_lr,
    target_shape=img.shape,
    scale=SR_SCALE,
    sigma=LR_NOISE_SIGMA,
    model=model,
    rho=ADMM_RHO,
    n_iter=ADMM_ITERS,
    n_cg=CG_ITERS,
    gt=img,
    verbose=True
)

psnr_final = psnr(img, x_hat, data_range=1.0)
print(f"\n✓ Final PnP-ADMM PSNR: {psnr_final:.2f} dB")
print(f"  Improvement over bicubic: +{psnr_final - psnr_bicubic:.2f} dB")

In [ ]:
# ============================================================
# 14. Visualize Results
# ============================================================

# Calculate error maps
error_bicubic = np.abs(img - bicubic)
error_pnp = np.abs(img - x_hat)

# Create comparison plot
fig, ax = plt.subplots(2, 3, figsize=(16, 11))

# Row 1: Images
ax[0, 0].imshow(img, cmap="gray", vmin=0, vmax=1)
ax[0, 0].set_title("Ground Truth HR", fontsize=14, fontweight='bold')
ax[0, 0].axis("off")

ax[0, 1].imshow(y_lr, cmap="gray", vmin=0, vmax=1)
ax[0, 1].set_title("Observed LR", fontsize=14, fontweight='bold')
ax[0, 1].axis("off")

ax[0, 2].imshow(bicubic, cmap="gray", vmin=0, vmax=1)
ax[0, 2].set_title(f"Bicubic\nPSNR: {psnr_bicubic:.2f} dB", fontsize=14, fontweight='bold')
ax[0, 2].axis("off")

# Row 2: Results and errors
ax[1, 0].imshow(x_hat, cmap="gray", vmin=0, vmax=1)
ax[1, 0].set_title(f"PnP-ADMM SR\nPSNR: {psnr_final:.2f} dB\n(+{psnr_final - psnr_bicubic:.2f} dB)", 
                   fontsize=14, fontweight='bold', color='green')
ax[1, 0].axis("off")

im1 = ax[1, 1].imshow(error_bicubic, cmap="hot")
ax[1, 1].set_title("Error: Bicubic", fontsize=14, fontweight='bold')
ax[1, 1].axis("off")
plt.colorbar(im1, ax=ax[1, 1], fraction=0.046)

im2 = ax[1, 2].imshow(error_pnp, cmap="hot")
ax[1, 2].set_title("Error: PnP-ADMM", fontsize=14, fontweight='bold')
ax[1, 2].axis("off")
plt.colorbar(im2, ax=ax[1, 2], fraction=0.046)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 15. Convergence Plots
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PSNR progression
if len(history["psnr"]) > 0:
    axes[0].plot(history["psnr"], marker="o", linewidth=2, markersize=6)
    axes[0].axhline(y=psnr_bicubic, color='r', linestyle='--', label='Bicubic baseline')
    axes[0].set_xlabel("Iteration", fontsize=12)
    axes[0].set_ylabel("PSNR (dB)", fontsize=12)
    axes[0].set_title("PSNR Progression", fontsize=14, fontweight='bold')
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

# Residual progression
axes[1].plot(history["residual"], marker="o", linewidth=2, markersize=6, color='orange')
axes[1].set_xlabel("Iteration", fontsize=12)
axes[1].set_ylabel("||x - v|| (Primal Residual)", fontsize=12)
axes[1].set_title("Convergence", fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 16. Save Results
# ============================================================

# Save the super-resolved image
output_path = "/kaggle/working/pnp_admm_result.png"
io.imsave(output_path, (np.clip(x_hat, 0, 1) * 255).astype(np.uint8))
print(f"✓ Saved SR result to: {output_path}")

# Save comparison
comparison_path = "/kaggle/working/comparison.png"
fig_comp = plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.imshow(y_lr, cmap='gray')
plt.title('LR Input')
plt.axis('off')
plt.subplot(1, 3, 2)
plt.imshow(bicubic, cmap='gray')
plt.title(f'Bicubic ({psnr_bicubic:.2f} dB)')
plt.axis('off')
plt.subplot(1, 3, 3)
plt.imshow(x_hat, cmap='gray')
plt.title(f'PnP-ADMM ({psnr_final:.2f} dB)')
plt.axis('off')
plt.tight_layout()
plt.savefig(comparison_path, dpi=150, bbox_inches='tight')
plt.close()
print(f"✓ Saved comparison to: {comparison_path}")

print("\n" + "="*60)
print("  SUPER-RESOLUTION COMPLETE!")
print("="*60)
print(f"  Input: {os.path.basename(test_image_path)}")
print(f"  Bicubic PSNR: {psnr_bicubic:.2f} dB")
print(f"  PnP-ADMM PSNR: {psnr_final:.2f} dB")
print(f"  Improvement: +{psnr_final - psnr_bicubic:.2f} dB")
print("="*60)

## How to Process Multiple Images

To process all uploaded images, run this cell:

In [ ]:
# ============================================================
# 17. Process All Images (Optional)
# ============================================================
for idx, img_path in enumerate(all_images):
    print(f"\n{'='*60}")
    print(f"Processing image {idx+1}/{len(all_images)}: {os.path.basename(img_path)}")
    print('='*60)
     
    # Load and process
    img = read_image_gray(img_path)
     
    # Resize if needed
    if img.shape[0] > MAX_IMAGE_SIZE or img.shape[1] > MAX_IMAGE_SIZE:
        scale_h = MAX_IMAGE_SIZE / img.shape[0]
        scale_w = MAX_IMAGE_SIZE / img.shape[1]
        resize_factor = min(scale_h, scale_w)
        img = zoom(img, resize_factor, order=3).astype(np.float32)
        img = np.clip(img, 0, 1)
     
    # Adjust dimensions
    H, W = img.shape
    H_adj = (H // SR_SCALE) * SR_SCALE
    W_adj = (W // SR_SCALE) * SR_SCALE
    img = img[:H_adj, :W_adj]
     
    # Create LR
    np.random.seed(idx)
    y_lr = forward_SR(img, SR_SCALE, BLUR_KERNEL)
    noise = np.random.randn(*y_lr.shape).astype(np.float32) * LR_NOISE_SIGMA
    y_lr = np.clip(y_lr + noise, 0, 1)
     
    # Run SR
    x_hat, _ = pnp_admm_sr(
        y_lr=y_lr,
        target_shape=img.shape,
        scale=SR_SCALE,
        sigma=LR_NOISE_SIGMA,
        model=model,
        rho=ADMM_RHO,
        n_iter=ADMM_ITERS,
        n_cg=CG_ITERS,
        gt=img,
        verbose=False
    )
     
    # Save
    output_name = f"/kaggle/working/sr_result_{idx:03d}.png"
    io.imsave(output_name, (np.clip(x_hat, 0, 1) * 255).astype(np.uint8))
    print(f"✓ Saved: {output_name}")